In [1]:
from pathlib import Path
import pandas as pd

project_folder = Path(r"D:\electricity-tariff-optimisation")
processed_folder = project_folder / "data" / "processed"

dashboard_folder = project_folder / "data" / "dashboard"
dashboard_folder.mkdir(parents=True, exist_ok=True)

source_files = {
    "customer_results": (
        processed_folder / "customer_tariff_results_2013.csv"
    ),
    "design_features": (
        processed_folder / "customer_design_features_2013.csv"
    ),
    "daily_profiles": (
        processed_folder / "customer_design_daily_profiles_2013.csv"
    ),
    "design_clusters": (
        processed_folder / "customer_design_cluster_assignments_2013.csv"
    ),
    "period_comparison": (
        processed_folder / "customer_profile_h1_h2_comparison_2013.csv"
    ),
    "cost_stress": (
        processed_folder / "validation_protected_tariff_cost_stress_2013.csv"
    ),
    "monitoring": (
        project_folder / "reports" / "pipeline_runs"
        / "20261029T025921Z_333b1572"
        / "customer_monitoring.csv"
    ),
}

dashboard_sources = {}

for name, file_path in source_files.items():
    assert file_path.is_file(), f"Missing source: {file_path}"
    frame = pd.read_csv(file_path)
    dashboard_sources[name] = frame

    print(f"\n{name}: {len(frame):,} rows")
    print("Columns:", ", ".join(frame.columns))

print("\nDashboard source files loaded.")
print("No dashboard export tables have been created yet.")


customer_results: 248 rows
Columns: household_id, assigned_tariff, energy_kwh, baseline_charge_gbp, tou_charge_gbp, assigned_charge_gbp, saving_gbp, saving_pct, variable_cost_gbp, usage_segment, assigned_tou, paying_more, above_increase_limit, rebate_gbp, protected_charge_gbp, protected_saving_gbp, protected_saving_pct, review_flag

design_features: 248 rows
Columns: household_id, design_kwh, peak_share_pct, shoulder_share_pct, off_peak_share_pct, review_flag, usage_segment

daily_profiles: 23,808 rows
Columns: household_id, day_type, slot, mean_energy_kwh, observed_intervals, time_label

design_clusters: 248 rows
Columns: household_id, design_cluster, profile_group, months_evaluated, months_matching_design, monthly_agreement_pct

period_comparison: 248 rows
Columns: household_id, design_cluster, validation_cluster, design_profile_group, validation_profile_group, same_group

cost_stress: 4 rows
Columns: cost_increase_pct, revenue_after_rebate_gbp, variable_cost_gbp, contribution_gbp, 

In [2]:
import numpy as np

customer_results = dashboard_sources["customer_results"].copy()
expected_ids = set(customer_results["household_id"])

# Validate household keys before merging.
for name in [
    "customer_results",
    "design_features",
    "design_clusters",
    "period_comparison",
    "monitoring",
]:
    frame = dashboard_sources[name]
    assert frame["household_id"].notna().all(), name
    assert frame["household_id"].is_unique, name
    assert set(frame["household_id"]) == expected_ids, name

design_features = dashboard_sources["design_features"][
    [
        "household_id",
        "design_kwh",
        "peak_share_pct",
        "shoulder_share_pct",
        "off_peak_share_pct",
    ]
].rename(columns={
    "peak_share_pct": "design_peak_share_pct",
    "shoulder_share_pct": "design_shoulder_share_pct",
    "off_peak_share_pct": "design_off_peak_share_pct",
})

profile_comparison = dashboard_sources["period_comparison"][
    [
        "household_id",
        "design_profile_group",
        "validation_profile_group",
    ]
]

monitoring_flags = dashboard_sources["monitoring"][
    [
        "household_id",
        "bill_increase_pct",
        "above_review_threshold",
        "profile_changed",
        "review_required",
        "review_reasons",
    ]
].rename(columns={
    "bill_increase_pct": "validation_bill_increase_pct",
})

customer_dashboard = (
    customer_results
    .merge(
        design_features,
        on="household_id",
        validate="one_to_one",
    )
    .merge(
        profile_comparison,
        on="household_id",
        validate="one_to_one",
    )
    .merge(
        monitoring_flags,
        on="household_id",
        validate="one_to_one",
    )
    .rename(columns={
        "energy_kwh": "validation_energy_kwh",
        "saving_gbp": "validation_saving_before_rebate_gbp",
        "saving_pct": "validation_saving_before_rebate_pct",
        "assigned_charge_gbp": "validation_charge_before_rebate_gbp",
        "baseline_charge_gbp": "validation_baseline_charge_gbp",
        "tou_charge_gbp": "validation_tou_charge_gbp",
        "variable_cost_gbp": "validation_variable_cost_gbp",
        "rebate_gbp": "scenario_rebate_gbp",
        "protected_charge_gbp": "scenario_protected_charge_gbp",
        "protected_saving_gbp": "scenario_protected_saving_gbp",
        "protected_saving_pct": "scenario_protected_saving_pct",
        "review_flag": "retrospective_quality_review_flag",
    })
)

saving = customer_dashboard["validation_saving_before_rebate_gbp"]
customer_dashboard["validation_charge_outcome"] = np.select(
    [saving > 1e-6, saving < -1e-6],
    ["Paying less", "Paying more"],
    default="Unchanged",
)

customer_dashboard["scenario_protected_bill_increase_pct"] = (
    -customer_dashboard["scenario_protected_saving_pct"]
)

# Reconcile both charge scenarios.
assert len(customer_dashboard) == 248
assert customer_dashboard["household_id"].is_unique

assert np.allclose(
    customer_dashboard["validation_baseline_charge_gbp"]
    - customer_dashboard["validation_charge_before_rebate_gbp"],
    customer_dashboard["validation_saving_before_rebate_gbp"],
    rtol=0,
    atol=1e-6,
)

assert np.allclose(
    customer_dashboard["validation_charge_before_rebate_gbp"]
    - customer_dashboard["scenario_rebate_gbp"],
    customer_dashboard["scenario_protected_charge_gbp"],
    rtol=0,
    atol=1e-6,
)

assert np.allclose(
    customer_dashboard["validation_baseline_charge_gbp"]
    - customer_dashboard["scenario_protected_charge_gbp"],
    customer_dashboard["scenario_protected_saving_gbp"],
    rtol=0,
    atol=1e-6,
)

output_file = dashboard_folder / "customer_dashboard_2013.csv"
customer_dashboard.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig",
)

print("Customer dashboard rows:", len(customer_dashboard))
print("Household keys and charge reconciliation: PASS")
print("\nValidation outcomes before rebates:")
print(
    customer_dashboard["validation_charge_outcome"]
    .value_counts().to_string()
)
print(
    "\nSaving before rebates (GBP): "
    f"{saving.sum():,.4f}"
)
print(
    "Protected scenario saving (GBP): "
    f"{customer_dashboard['scenario_protected_saving_gbp'].sum():,.4f}"
)
print("Saved:", output_file)

Customer dashboard rows: 248
Household keys and charge reconciliation: PASS

Validation outcomes before rebates:
validation_charge_outcome
Paying less    213
Unchanged       28
Paying more      7

Saving before rebates (GBP): 3,925.4984
Protected scenario saving (GBP): 3,929.2839
Saved: D:\electricity-tariff-optimisation\data\dashboard\customer_dashboard_2013.csv


In [3]:
profile_dashboard = dashboard_sources["daily_profiles"].copy()

profile_key = ["household_id", "day_type", "slot"]
assert not profile_dashboard.duplicated(profile_key).any()
assert set(profile_dashboard["household_id"]) == expected_ids
assert set(profile_dashboard["day_type"]) == {"Weekday", "Weekend"}
assert profile_dashboard["slot"].isin(range(48)).all()

profile_counts = profile_dashboard.groupby(
    ["household_id", "day_type"]
).size()
assert profile_counts.eq(48).all()

assert np.isfinite(
    profile_dashboard["mean_energy_kwh"].to_numpy()
).all()
assert profile_dashboard["mean_energy_kwh"].ge(0).all()
assert profile_dashboard["observed_intervals"].gt(0).all()

# Match the combined weekday/weekend normalisation used for clustering.
profile_means = profile_dashboard.groupby("household_id")[
    "mean_energy_kwh"
].transform("mean")

assert profile_means.gt(0).all()

profile_dashboard["normalised_mean_energy"] = (
    profile_dashboard["mean_energy_kwh"] / profile_means
)
profile_dashboard["local_hour"] = profile_dashboard["slot"] / 2
profile_dashboard["day_type_sort"] = (
    profile_dashboard["day_type"].map({"Weekday": 1, "Weekend": 2})
)
profile_dashboard["period"] = "Design: Jan–Jun 2013"

assert np.allclose(
    profile_dashboard.groupby("household_id")[
        "normalised_mean_energy"
    ].mean(),
    1.0,
)

profile_dashboard.to_csv(
    dashboard_folder / "design_profile_dashboard_2013.csv",
    index=False,
    encoding="utf-8-sig",
)

# Portfolio-level scenarios: keep separate from household-level tables.
cost_stress_dashboard = dashboard_sources["cost_stress"].copy()

assert cost_stress_dashboard["cost_increase_pct"].is_unique

numeric_columns = [
    "cost_increase_pct",
    "revenue_after_rebate_gbp",
    "variable_cost_gbp",
    "contribution_gbp",
    "contribution_margin_pct",
]
assert np.isfinite(
    cost_stress_dashboard[numeric_columns].to_numpy(dtype=float)
).all()

assert np.allclose(
    cost_stress_dashboard["revenue_after_rebate_gbp"]
    - cost_stress_dashboard["variable_cost_gbp"],
    cost_stress_dashboard["contribution_gbp"],
    rtol=0,
    atol=1e-6,
)

assert np.allclose(
    100 * cost_stress_dashboard["contribution_gbp"]
    / cost_stress_dashboard["revenue_after_rebate_gbp"],
    cost_stress_dashboard["contribution_margin_pct"],
    rtol=0,
    atol=1e-6,
)

cost_stress_dashboard["margin_target_pct"] = 25.0
cost_stress_dashboard["scenario_basis"] = (
    "Post-validation rebate scenario"
)

cost_stress_dashboard.to_csv(
    dashboard_folder / "cost_stress_dashboard_2013.csv",
    index=False,
    encoding="utf-8-sig",
)

print("Design profile rows:", len(profile_dashboard))
print("Profile completeness and normalisation: PASS")
print("Cost stress scenarios:", len(cost_stress_dashboard))
print("Cost stress reconciliation: PASS")
print("Both dashboard tables saved.")

Design profile rows: 23808
Profile completeness and normalisation: PASS
Cost stress scenarios: 4
Cost stress reconciliation: PASS
Both dashboard tables saved.


In [4]:
monthly_folder = project_folder / "reports" / "monthly_monitoring"

monthly_dashboard = pd.read_csv(
    monthly_folder / "customer_monthly_monitoring_2013.csv"
)
household_review = pd.read_csv(
    monthly_folder / "household_monthly_review_2013.csv"
)

customer_file = dashboard_folder / "customer_dashboard_2013.csv"
customers = pd.read_csv(customer_file)

assert len(monthly_dashboard) == 1488
assert not monthly_dashboard.duplicated(
    ["household_id", "month_utc"]
).any()
assert set(monthly_dashboard["household_id"]) == set(
    customers["household_id"]
)

monthly_dashboard["month_start"] = pd.to_datetime(
    monthly_dashboard["month_utc"] + "-01",
    format="%Y-%m-%d",
)
monthly_dashboard["charge_basis"] = "Before rebates"
monthly_dashboard["review_threshold_pct"] = 2.0

monthly_dashboard.to_csv(
    dashboard_folder / "monthly_monitoring_dashboard_2013.csv",
    index=False,
    encoding="utf-8-sig",
    date_format="%Y-%m-%d",
)

review_columns = [
    "months_paying_more",
    "months_above_threshold",
    "months_profile_changed",
    "months_profile_unavailable",
    "worst_month_increase_pct",
    "longest_paying_more_streak",
    "h2_saver_with_loss_month",
]

assert household_review["household_id"].is_unique
assert set(household_review["household_id"]) == set(
    customers["household_id"]
)

# Allow this cell to be rerun without creating duplicate columns.
customers = customers.drop(
    columns=review_columns,
    errors="ignore",
).merge(
    household_review[["household_id"] + review_columns],
    on="household_id",
    validate="one_to_one",
)

assert len(customers) == 248
assert customers["months_paying_more"].notna().all()

customers.to_csv(
    customer_file,
    index=False,
    encoding="utf-8-sig",
)

print("Monthly dashboard rows:", len(monthly_dashboard))
print("Updated customer dashboard rows:", len(customers))
print(
    "Households with any loss month:",
    int(customers["months_paying_more"].gt(0).sum()),
)
print(
    "Households with three consecutive loss months:",
    int(customers["longest_paying_more_streak"].ge(3).sum()),
)
print("Dashboard exports updated.")
print("Monthly flags are retrospective review indicators.")

Monthly dashboard rows: 1488
Updated customer dashboard rows: 248
Households with any loss month: 55
Households with three consecutive loss months: 10
Dashboard exports updated.
Monthly flags are retrospective review indicators.
